In [ ]:
import os
import json
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One provider described as pure data"""
    
    name: str
    """Name of the provider"""
    env_var: str
    """Environment variable name for the provider's API key"""
    is_free: bool
    """Whether the provider is free or not"""
    base_url: str|None
    """Base URL for the provider's API"""
    model: str|None
    """Model name for the provider's API"""
    
    
PROVIDERS=[
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/",
        model="openai/gpt-oss-120b"  
    ),
    # can i use one more geoq provider -> answer: yes, you can add another provider by creating a new instance of the Provider class and adding it to the PROVIDERS dictionary. For example:
    Provider(
        name="Groq2",
        env_var="GROQ2_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/",
        model="openai/gpt-oss-20b"  
    ) 
]

def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var): # check if the environment variable for the provider's API key is set
            return provider
        
        expected = ", ".join(provider.env_var for provider in PROVIDERS)
        raise RuntimeError(f"No provider key set. Please set one of the following environment variables: {expected}")
        
def build_client(provider: Provider):
    from groq import Groq
    
    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return Groq(api_key=api_key)
    else:
        return Groq(api_key=api_key, base_url=provider.base_url)
    
def have_any_key()->bool:
    return any(os.environ.get(provider.env_var) for provider in PROVIDERS)

def llm_reply(prompt: str, *, max_tokens: int = 400)->str:
    # send one user prompt...returns the assistant's text
    provider = select_provider()
    client = build_client(provider)
    response = client.chat.completions.create(
        model=provider.model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content 

In [2]:
WEATHER_DB = {
    "london": {"celsius": 22, "sky": "cloudy"},
    "paris": {"celsius": 24, "sky": "sunny"},
    "tallinn": {"celsius": 28, "sky": "sunny"},
    "moscow": {"celsius": 18, "sky": "rainy"},
    "tokyo": {"celsius": 30, "sky": "sunny"},
    "beijing": {"celsius": 26, "sky": "cloudy"},
    "new york": {"celsius": 28, "sky": "sunny"},
    "mumbai": {"celsius": 32, "sky": "rainy"},
    "cape town": {"celsius": 16, "sky": "cloudy"},
    "sydney": {"celsius": 20, "sky": "sunny"},
    "rio de janeiro": {"celsius": 22, "sky": "cloudy"},
    "cairo": {"celsius": 24, "sky": "sunny"},
    "mexico city": {"celsius": 20, "sky": "cloudy"},
}

In [3]:
def lookup_weather(location: str)->str:
    record = WEATHER_DB.get(location.lower())
    if record is None:
        return f"Sorry, I don't know the weather in {location}"
    return f"The weather in {location} is {record['celsius']} C and {record['sky']}."
    

In [4]:
# what the tool is capable of and how to use the tool
# this schema will be given to LLM
weather_schema = {
    "type":"function",
    "function":{
        "name":"lookup_weather", 
        "description":"Look up the weather for a location.",
        "parameters":{
            "type":"object",
            "properties":{
                "location":{"type":"string","description":"The location to look up the waether for."}
            },
            "required":["location"]
        }
    }
}

TOOLS = {
    "lookup_weather":lookup_weather
    # Other tools can be define
}

In [5]:

def ask_llm_with_tool(prompt: str, *, max_tokens: int = 400) -> str:
    import json

    provider = select_provider()
    client = build_client(provider)

    messages = [{"role": "user", "content": prompt}]
    
    while True:
        response = client.chat.completions.create(
            model = provider.model,
            messages=messages,
            max_tokens=max_tokens,
            tools=[weather_schema],
            # tool_choice="auto" 
        )
        msg = response.choices[0].message
        
        if not msg.tool_calls:
            return msg.content
        
        messages.append({
            "role":"assistant",
            "content": msg.content,
            "tool_calls":msg.tool_calls
        })
        
        # llm is asking us to call a tool
        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            tool_call_id = tool_call.id
            
            if tool_name not in TOOLS:
                raise ValueError(f"Tool {tool_name} not found.")
            tool = TOOLS[tool_name]
            result = tool(**tool_args)
            messages.append(
                {
                    "role":"tool",
                    "tool_call_id":tool_call_id,
                    "content":result
                }
            )
        print("All tool calls done..going back to LLM")

In [6]:
msg = ask_llm_with_tool("What is the weather in Tokyo")
print("Final response from llm:", msg)


All tool calls done..going back to LLM
Final response from llm: The current weather in Tokyo is sunny with a temperature of about 30 °C.
